# Non-linear (XGBoost) destination probe -- run all cells top to bottom

**What this is for.** The paper reports a *linear* probe inside SmolVLA's action expert. It
decodes the named destination on one checkpoint and fails on the other. A linear null has two
readings that mean very different things:

* the destination is **not encoded** at that site, or
* it **is** encoded, but not in a form a hyperplane can read.

The paper cannot currently separate them. This run fits gradient-boosted trees at every site on
the same split, same standardisation, same accuracy readout. Only the hypothesis class changes,
so any difference is attributable to linearity and nothing else.

**Reading the result** (the last cell prints it):

* boosted approximately equals linear -> the linear null is a **genuine absence**. This is the
  expected outcome and it strengthens the paper.
* boosted clearly beats linear on expert sites -> the destination is **present but not linearly
  readable**. That is a different claim from the one the paper makes, and a more interesting one.

Either outcome is useful. Nothing here should be tuned to produce a particular answer.

**Cost.** Roughly 30-60 minutes for both checkpoints on one GPU. Almost all of it is forward
passes; the probe fits are cheap. Activations are cached to disk, so any later probe variant
costs seconds instead of a full rerun.

**Nothing is saved unless it is pushed.** On a temporary machine, do the credentials cell.
Otherwise use the download fallback in section 8 before the machine is reclaimed.

## 1. Confirm there is a real GPU before spending time on it

In [ ]:
!nvidia-smi

## 2. Find or clone the repo

Works whether you already cloned it or are starting cold. If this notebook already lives inside
the repo, it walks up to the root rather than nesting a second copy.

In [ ]:
import os
import pathlib
import subprocess

REPO = "https://github.com/mzkaell/vla-where-does-language-die.git"
BRANCH = "phase1-scaffold"


def find_repo_root(start: pathlib.Path):
    for d in [start, *start.parents]:
        if (d / "pyproject.toml").exists() and (d / "src" / "models").exists():
            return d
    return None


root = find_repo_root(pathlib.Path.cwd())
if root is None:
    target = pathlib.Path.home() / "vla-where-does-language-die"
    if not target.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO, str(target)], check=True)
    root = target

os.chdir(root)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=False)
print("repo root:", root)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 3. GitHub credentials (optional, but this is how results survive)

Uses `getpass`, so the token is never written into this notebook or its saved output. A
fine-grained PAT with *Contents: read and write* on this repo only is enough.

Skip it if you prefer. The run still works, and section 8 has a download fallback.

In [ ]:
import getpass
import os
import subprocess

token = getpass.getpass("GitHub token (blank to skip pushing): ").strip()
if token:
    subprocess.run(
        ["git", "remote", "set-url", "origin",
         f"https://{token}@github.com/mzkaell/vla-where-does-language-die.git"],
        check=True,
    )
    subprocess.run(["git", "config", "user.name", "mzkaell"], check=True)
    subprocess.run(["git", "config", "user.email", "schmalzmichael50@gmail.com"], check=True)
    os.environ["GIT_PUSH"] = "1"
    # Verify now rather than discovering it fails after an hour of compute.
    ok = subprocess.run(["git", "push", "--dry-run", "origin", "HEAD"],
                        capture_output=True, text=True)
    print("push check:", "OK" if ok.returncode == 0 else "FAILED\n" + ok.stderr)
else:
    os.environ["GIT_PUSH"] = "0"
    print("not pushing -- use the download fallback in section 8")

## 4. Install (~5 min)

`xgboost` was added as a dependency for exactly this experiment, so an environment built
before this week will not have it.

In [ ]:
!pip install -q -e ".[vla,dev]"
!pip install -q "xgboost>=2.0"

import torch
import xgboost

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("xgboost", xgboost.__version__)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU VISIBLE")

## 5. Sanity-check the probe before the long run

These are the tests that make a null result meaningful. The boosted probe must recover
XOR-structured labels a linear probe cannot see; if it cannot, a null from it says nothing
about the model, only about the probe. It must also stay near chance on shuffled labels.

In [ ]:
!python -m pytest -q -p no:warnings tests/test_stats.py -k nonlinear

## 6. Launch both checkpoints in the background

`nohup` detaches the process, so closing the tab or losing the connection will not kill it.
The script skips any checkpoint that already has results, so re-running this cell after an
interruption resumes rather than restarting.

`--cache-activations` writes pooled activations to disk. The forward passes are the entire cost
and are identical for every probe family, so this makes follow-up probes nearly free.

In [ ]:
%%bash
cat > run_xgb.sh <<'SH'
set -euo pipefail
for CKPT in k1000dai/smolvla_libero_finetune k1000dai/smolvla_libero_scratch_80k; do
  NAME="probe_nl_$(basename "$CKPT" | sed 's/smolvla_libero_//')"
  if [ -f "results/$NAME/metrics.json" ]; then
    echo "== $NAME already done, skipping"
    continue
  fi
  echo "== $NAME starting at $(date)"
  python scripts/run_probe.py \
      --checkpoint "$CKPT" --n-states 150 --device cuda \
      --nonlinear --cache-activations --run-id "$NAME"
  if [ "${GIT_PUSH:-0}" = "1" ]; then
    git add -f "results/$NAME/metrics.json" "results/$NAME/config.yaml"
    git commit -m "Non-linear probe: $NAME" || true
    git push origin HEAD || echo "PUSH FAILED -- use the download fallback"
  fi
done
echo "== all done at $(date)"
SH
nohup bash run_xgb.sh >> xgb_probe.log 2>&1 &
echo "launched -- safe to close the tab"

## 7. Monitor (re-run this cell whenever)

In [ ]:
!tail -30 xgb_probe.log

## 8. The result

This prints the comparison the experiment exists to make. **Send this output back.**

In [ ]:
import json
import pathlib

import numpy as np

for name in ["probe_nl_finetune", "probe_nl_scratch_80k"]:
    p = pathlib.Path("results") / name / "metrics.json"
    if not p.exists():
        print(f"{name}: not finished yet")
        continue
    d = json.loads(p.read_text())
    if "nonlinear" not in d:
        print(f"{name}: no nonlinear block -- was --nonlinear passed?")
        continue
    nl = d["nonlinear"]
    exp = [s for s in d["sites"] if s["site"].startswith("expert.")]
    deltas = [nl[s["site"]]["acc_novel"] - s["acc_novel"] for s in exp]
    gained = [s for s, dd in zip(exp, deltas) if dd > 0.10]
    # A boosted probe that gains only where its OWN shuffled control also rises is fitting
    # noise, not finding structure. Report both so one cannot be mistaken for the other.
    real = [s for s in gained if nl[s["site"]]["acc_shuffled"] < 0.35]
    print(f"===== {name} =====")
    print(f"  chance                           : {exp[0]['chance']:.3f}")
    print(f"  expert sites                     : {len(exp)}")
    print(f"  linear  max acc (novel)          : {max(s['acc_novel'] for s in exp):.3f}")
    print(f"  boosted max acc (novel)          : {max(nl[s['site']]['acc_novel'] for s in exp):.3f}")
    print(f"  mean(boosted - linear)           : {np.mean(deltas):+.3f}")
    print(f"  sites where boosted beats by .10 : {len(gained)}"
          f"  (of which shuffled stays low: {len(real)})")
    print("  ->", "LINEAR NULL IS GENUINE" if len(real) <= 2 else
          "NON-LINEARLY DECODABLE -- flag this, it changes a claim")
    print()

## 9. Before the machine is reclaimed: confirm results are off-machine

Do not trust the log. Open the repo on GitHub and check that `results/probe_nl_*` exist. If the
push failed, download the bundle below through the Jupyter file browser.

In [ ]:
!git log --oneline -3
!ls -1 results/ | grep probe_nl || echo "no probe_nl results yet"
!tar czf xgb_results.tar.gz results/probe_nl_* 2>/dev/null && ls -lh xgb_results.tar.gz
print("If the push failed: right-click xgb_results.tar.gz in the file browser -> Download.")